# NB20 — sklearn Pipelines: Production-Ready Workflow

> **Combine preprocessing + model into one object. Prevents data leakage. Easy to deploy.**

A Pipeline applies steps in sequence:
`StandardScaler → PolynomialFeatures → Ridge`

Why Pipeline and not manual steps?
- **No data leakage**: scaler is fitted only on train data, even inside cross-validation
- **One object to serialize and deploy**
- **GridSearchCV works over the whole pipeline**


In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_squared_error, r2_score
import warnings; warnings.filterwarnings('ignore')

housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target

# Introduce some missing values for realism
np.random.seed(42)
missing_mask = np.random.rand(*X.shape) < 0.03   # 3% missing
X_missing = X.copy()
X_missing[missing_mask] = np.nan

X_tr, X_te, y_tr, y_te = train_test_split(X_missing, y, test_size=0.2, random_state=0)
print(f"Train: {X_tr.shape}, Test: {X_te.shape}")
print(f"Missing values in train: {X_tr.isna().sum().sum()}")


In [ ]:
# Build a full pipeline
pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),       # step 1: fill missing
    ('scaler',  StandardScaler()),                        # step 2: standardise
    ('model',   Ridge(alpha=1.0)),                        # step 3: model
])

pipe.fit(X_tr, y_tr)
y_pred = pipe.predict(X_te)
print(f"Ridge Pipeline — Test R²:   {r2_score(y_te, y_pred):.4f}")
print(f"Ridge Pipeline — Test RMSE: {np.sqrt(mean_squared_error(y_te, y_pred)):.4f}")


In [ ]:
# GridSearchCV over the entire pipeline
from sklearn.model_selection import GridSearchCV

param_grid = {
    'model__alpha': [0.01, 0.1, 1, 10, 100],    # prefix = step name + __
}
grid = GridSearchCV(pipe, param_grid, cv=5, scoring='r2', n_jobs=-1)
grid.fit(X_tr, y_tr)

print(f"Best alpha: {grid.best_params_['model__alpha']}")
print(f"Best CV R²: {grid.best_score_:.4f}")
print(f"Test R²:    {grid.score(X_te, y_te):.4f}")

import pandas as pd
results = pd.DataFrame(grid.cv_results_)[['param_model__alpha','mean_test_score','std_test_score']]
print("\nAll results:"); print(results.to_string(index=False))


In [ ]:
# Compare multiple models in one grid search using a custom step
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.model_selection import GridSearchCV
import numpy as np

# Put model choice inside the param_grid
pipes = {
    'Ridge':      Pipeline([('imp', SimpleImputer()), ('sc', StandardScaler()), ('m', Ridge())]),
    'Lasso':      Pipeline([('imp', SimpleImputer()), ('sc', StandardScaler()), ('m', Lasso())]),
    'ElasticNet': Pipeline([('imp', SimpleImputer()), ('sc', StandardScaler()), ('m', ElasticNet())]),
}

alphas = np.logspace(-2, 2, 20)
print(f"{'Model':<15} {'Best alpha':>12} {'CV R²':>8} {'Test R²':>8}")
print("-"*50)
for name, pipe in pipes.items():
    gs = GridSearchCV(pipe, {'m__alpha': alphas}, cv=5, scoring='r2', n_jobs=-1)
    gs.fit(X_tr, y_tr)
    print(f"{name:<15} {gs.best_params_['m__alpha']:>12.4f} {gs.best_score_:>8.4f} {gs.score(X_te, y_te):>8.4f}")


In [ ]:
# Save and load the pipeline (production deployment)
import joblib, os

save_path = os.path.join(os.path.dirname(__file__) if '__file__' in dir() else '.', 'best_pipeline.pkl')
joblib.dump(grid.best_estimator_, save_path)
print(f"Saved pipeline to {save_path}")

loaded_pipe = joblib.load(save_path)
y_loaded    = loaded_pipe.predict(X_te)
from sklearn.metrics import r2_score
print(f"Loaded pipeline Test R²: {r2_score(y_te, y_loaded):.4f}  (should match above)")

# Make a single prediction from new data
new_obs = pd.DataFrame([X.median().values], columns=X.columns)
new_obs.iloc[0, 0] = np.nan   # introduce a missing value
pred = loaded_pipe.predict(new_obs)
print(f"\nPrediction for median house: ${pred[0]*100:.0f}k")


## Production checklist

| Step | Why |
|------|-----|
| Imputer inside Pipeline | Impute using train-set statistics only |
| StandardScaler inside Pipeline | Prevents leakage of test-set mean/std into CV |
| GridSearchCV on whole pipeline | Tunes hyperparameters without leakage |
| joblib.dump / load | Serialize for serving |
| Single predict() call | Applies all preprocessing automatically |

## Full curriculum summary

| NB | Topic | Key concept |
|----|-------|-------------|
| 01 | Intuition | Best-fit line minimises SSR |
| 02 | OLS derivation | β̂ = (XᵀX)⁻¹Xᵀy from calculus |
| 03 | From scratch | Pure Python implementation |
| 04 | R² | SS decomposition: SS_tot = SS_reg + SS_res |
| 05 | Inference | t-stats, p-values, confidence intervals |
| 06 | Assumptions | LINE: Linearity, Independence, Normality, Equal variance |
| 07 | Diagnostics | 4 standard plots, Cook's D |
| 08 | Multiple regression | Matrix form, β interpretation |
| 09 | Multicollinearity | VIF, correlation heatmap |
| 10 | Poly & interactions | Feature engineering |
| 11 | Ridge | L2 penalty, multicollinearity fix |
| 12 | Lasso | L1 penalty, automatic feature selection |
| 13 | ElasticNet | L1+L2, correlated features |
| 14 | Cross-validation | k-fold CV, learning curves, nested CV |
| 15 | Gradient Descent | GD & SGD from scratch |
| 16 | Robust regression | Huber, RANSAC, Theil-Sen |
| 17 | Quantile regression | Model any quantile, prediction intervals |
| 18 | Bayesian | Posterior distribution over coefficients |
| 19 | Time series | Trend, Fourier, lag features |
| 20 | Pipelines | Production-ready sklearn workflow |
